# PID Control from Scratch
## A Complete Guide to Proportional-Integral-Derivative Control for Robotics

This notebook provides a rigorous, from-scratch implementation of **PID control** — the most widely used feedback control strategy in engineering. We derive each component, implement a full PID controller, analyze stability, explore tuning methods, and apply the controller to two robotics problems.

**What you'll learn:**
- How proportional, integral, and derivative actions each address different aspects of tracking error
- The mathematics of PID control: continuous and discrete formulations
- Stability analysis using the Routh-Hurwitz criterion
- Systematic tuning via Ziegler-Nichols
- Application to cart-pole balancing and drone altitude hold
- Practical improvements: anti-windup, derivative filtering, derivative-on-measurement

**Prerequisites:** Ordinary differential equations, Laplace transforms (basics), state-space representations.

**References:**
- Ogata, *Modern Control Engineering*, 5th Ed., Prentice Hall, 2010.
- Astrom & Murray, *Feedback Systems: An Introduction for Scientists and Engineers*, Princeton University Press, 2008.
- Franklin, Powell & Emami-Naeini, *Feedback Control of Dynamic Systems*, 7th Ed., Pearson, 2015.

---
## 1. Introduction: Why Feedback Control?

### The Fundamental Problem

Consider a physical system (the **plant**) whose output $y(t)$ we want to track a desired reference $r(t)$. Without feedback, any disturbance, model uncertainty, or noise will cause the output to deviate from the reference with no corrective action.

**Feedback control** measures the output, computes the **error** $e(t) = r(t) - y(t)$, and applies a control input $u(t)$ to drive the error toward zero.

### Why PID?

The PID controller is the workhorse of industrial control. Over 90% of all control loops in practice use some form of PID. Its appeal:

| Property | Benefit |
|----------|--------|
| Simple structure | Only 3 tuning parameters |
| Model-free tuning | Can be tuned without a plant model |
| Robust performance | Works well for a wide range of plants |
| Intuitive interpretation | Each term has clear physical meaning |

### Plant Models for This Notebook

We will use two canonical plants:

**1. Mass-Spring-Damper:** A second-order linear system:

$$m\ddot{x} + b\dot{x} + kx = u(t)$$

with mass $m$, damping coefficient $b$, spring constant $k$, and control force $u$.

**2. Cart-Pole (Inverted Pendulum):** A classic nonlinear underactuated system — a pole balanced on a cart by applying horizontal force.

**3. Drone (1D Altitude):** Vertical dynamics governed by $m\ddot{z} = T - mg$.

---
## 2. Proportional Control

### The P-Controller

The simplest feedback controller applies a control effort proportional to the error:

$$u(t) = K_p \, e(t) = K_p \big(r(t) - y(t)\big)$$

where $K_p > 0$ is the **proportional gain**.

### Steady-State Error Analysis

For the mass-spring-damper with P-control, the closed-loop equation under a step input $r(t) = r_0$ is:

$$m\ddot{x} + b\dot{x} + kx = K_p(r_0 - x)$$
$$m\ddot{x} + b\dot{x} + (k + K_p)x = K_p \, r_0$$

At steady state ($\ddot{x} = \dot{x} = 0$):

$$(k + K_p) x_{ss} = K_p \, r_0 \implies x_{ss} = \frac{K_p}{k + K_p} r_0$$

The steady-state error is:

$$\boxed{e_{ss} = r_0 - x_{ss} = \frac{k}{k + K_p} r_0}$$

Key insight: for a type-0 plant (nonzero $k$), the P-controller **cannot** eliminate steady-state error. Increasing $K_p$ reduces $e_{ss}$ but never makes it zero, and high $K_p$ leads to oscillation.

### RK4 Integration

We implement the classical 4th-order Runge-Kutta method from scratch. For a system $\dot{\mathbf{x}} = f(\mathbf{x}, u)$:

$$\mathbf{k}_1 = f(\mathbf{x}_n, u_n)$$
$$\mathbf{k}_2 = f\!\left(\mathbf{x}_n + \frac{dt}{2}\mathbf{k}_1,\, u_n\right)$$
$$\mathbf{k}_3 = f\!\left(\mathbf{x}_n + \frac{dt}{2}\mathbf{k}_2,\, u_n\right)$$
$$\mathbf{k}_4 = f(\mathbf{x}_n + dt \cdot \mathbf{k}_3,\, u_n)$$
$$\mathbf{x}_{n+1} = \mathbf{x}_n + \frac{dt}{6}(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

np.random.seed(42)

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================

# Mass-Spring-Damper parameters
MSD_MASS = 1.0          # Mass [kg]
MSD_DAMPING = 0.5       # Damping coefficient [N*s/m]
MSD_SPRING = 2.0        # Spring constant [N/m]

# Cart-Pole parameters
CP_CART_MASS = 1.0      # Cart mass M [kg]
CP_POLE_MASS = 0.1      # Pole mass m [kg]
CP_POLE_LENGTH = 0.5    # Pole half-length l [m]
CP_GRAVITY = 9.81       # Gravitational acceleration [m/s^2]

# Drone parameters
DRONE_MASS = 1.0        # Drone mass [kg]
DRONE_GRAVITY = 9.81    # Gravitational acceleration [m/s^2]

# Simulation defaults
DT = 0.001              # Time step [s]
T_SIM = 10.0            # Default simulation time [s]

# Color scheme
COLORS = {
    'blue': 'steelblue',
    'red': 'coral',
    'green': 'seagreen',
    'yellow': 'goldenrod',
    'purple': 'mediumpurple',
    'orange': 'darkorange',
}

In [ ]:
def rk4_step(f, state, u, dt):
    """
    Perform a single RK4 integration step.

    Args:
        f: Callable, dynamics function f(state, u) -> state_dot.
           Shape: state (N,) -> (N,)
        state: Current state vector. Shape: (N,)
        u: Control input (scalar or array).
        dt: Time step [s].

    Returns:
        next_state: State after one time step. Shape: (N,)
    """
    k1 = f(state, u)
    k2 = f(state + 0.5 * dt * k1, u)
    k3 = f(state + 0.5 * dt * k2, u)
    k4 = f(state + dt * k3, u)
    return state + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def mass_spring_damper(state, u, m=MSD_MASS, b=MSD_DAMPING, k=MSD_SPRING):
    """
    Dynamics of a mass-spring-damper system: m*x'' + b*x' + k*x = u.

    Args:
        state: [x, x_dot]. Shape: (2,)
        u: Control force [N]. Scalar.
        m: Mass [kg].
        b: Damping coefficient [N*s/m].
        k: Spring constant [N/m].

    Returns:
        state_dot: [x_dot, x_ddot]. Shape: (2,)
    """
    x, x_dot = state
    x_ddot = (u - b * x_dot - k * x) / m
    return np.array([x_dot, x_ddot])


def simulate_plant(plant_func, controller_func, r, dt=DT, T=T_SIM, x0=None):
    """
    Simulate a plant with a controller using RK4 integration.

    Args:
        plant_func: Callable(state, u) -> state_dot. The plant dynamics.
        controller_func: Callable(error, state, t) -> u. The controller.
        r: Reference value (scalar, step input).
        dt: Time step [s].
        T: Total simulation time [s].
        x0: Initial state. Shape: (N,). Defaults to zeros.

    Returns:
        t_hist: Time array. Shape: (M,)
        x_hist: State history. Shape: (M, N)
        u_hist: Control history. Shape: (M,)
    """
    n_steps = int(T / dt)
    if x0 is None:
        # Infer state dimension by calling plant_func with zeros
        test_dot = plant_func(np.zeros(2), 0.0)
        x0 = np.zeros(len(test_dot))

    state = np.array(x0, dtype=float)
    n_states = len(state)

    t_hist = np.zeros(n_steps + 1)
    x_hist = np.zeros((n_steps + 1, n_states))
    u_hist = np.zeros(n_steps + 1)

    x_hist[0] = state

    for i in range(n_steps):
        t = i * dt
        y = state[0]  # output is first state variable
        error = r - y
        u = controller_func(error, state, t)
        u_hist[i] = u

        state = rk4_step(plant_func, state, u, dt)
        t_hist[i + 1] = (i + 1) * dt
        x_hist[i + 1] = state

    # Final control
    u_hist[-1] = u_hist[-2]

    return t_hist, x_hist, u_hist

In [ ]:
# --- Demonstrate P-only control on mass-spring-damper ---
R_STEP = 1.0  # Step reference

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

kp_values = [2.0, 5.0, 10.0, 50.0]
colors_list = [COLORS['blue'], COLORS['red'], COLORS['green'], COLORS['yellow']]

for kp, color in zip(kp_values, colors_list):
    def p_controller(error, state, t, _kp=kp):
        return _kp * error

    t, x, u = simulate_plant(mass_spring_damper, p_controller, R_STEP, T=T_SIM)

    # Theoretical steady-state error
    ess_theory = MSD_SPRING / (MSD_SPRING + kp) * R_STEP
    ess_actual = R_STEP - x[-1, 0]

    axes[0].plot(t, x[:, 0], color=color,
                 label=f'$K_p={kp}$ (ess={ess_actual:.3f})')
    axes[1].plot(t, R_STEP - x[:, 0], color=color,
                 label=f'$K_p={kp}$')

axes[0].axhline(y=R_STEP, color='black', linestyle='--', alpha=0.5, label='Reference')
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Position $x(t)$')
axes[0].set_title('P-Control: Step Response')
axes[0].legend(fontsize=10)

axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Error $e(t)$')
axes[1].set_title('P-Control: Tracking Error')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

# Verify steady-state error formula
print("Steady-state error verification (P-control on mass-spring-damper):")
for kp in kp_values:
    def p_ctrl(error, state, t, _kp=kp):
        return _kp * error
    _, x_sim, _ = simulate_plant(mass_spring_damper, p_ctrl, R_STEP, T=20.0)
    ess_theory = MSD_SPRING / (MSD_SPRING + kp) * R_STEP
    ess_sim = R_STEP - x_sim[-1, 0]
    match = abs(ess_theory - ess_sim) < 0.01
    status = "PASS" if match else "FAIL"
    print(f"  Kp={kp:5.1f}: theory={ess_theory:.4f}, sim={ess_sim:.4f} [{status}]")

---
## 3. Full PID Formulation

### Continuous-Time PID

The full PID control law combines three terms:

$$\boxed{u(t) = K_p \, e(t) + K_i \int_0^t e(\tau)\,d\tau + K_d \, \dot{e}(t)}$$

| Term | Action | Effect |
|------|--------|--------|
| $K_p \, e(t)$ | Proportional | Reacts to present error — provides stiffness |
| $K_i \int e \, d\tau$ | Integral | Reacts to accumulated past error — eliminates steady-state error |
| $K_d \, \dot{e}$ | Derivative | Reacts to rate of change — provides damping, reduces overshoot |

### Transfer Function Form

In the Laplace domain:

$$C(s) = K_p + \frac{K_i}{s} + K_d s = \frac{K_d s^2 + K_p s + K_i}{s}$$

The integral term introduces a pole at $s = 0$, which increases the system type by one and eliminates steady-state error to step inputs.

### Discrete-Time Implementation

For digital implementation with sampling period $\Delta t$:

- **Integral**: Rectangular (Euler) approximation: $I[k] = I[k-1] + e[k] \cdot \Delta t$
- **Derivative**: Backward difference: $D[k] = \frac{e[k] - e[k-1]}{\Delta t}$

$$\boxed{u[k] = K_p \, e[k] + K_i \sum_{j=0}^{k} e[j] \, \Delta t + K_d \frac{e[k] - e[k-1]}{\Delta t}}$$

In [ ]:
class PIDController:
    """
    Discrete-time PID controller.

    Implements the standard positional PID algorithm:
        u[k] = Kp * e[k] + Ki * integral(e) * dt + Kd * (e[k] - e[k-1]) / dt

    Args:
        Kp: Proportional gain.
        Ki: Integral gain.
        Kd: Derivative gain.
        dt: Sampling period [s].
        u_min: Minimum control output (for anti-windup). Default: -inf.
        u_max: Maximum control output (for anti-windup). Default: +inf.
        derivative_on_measurement: If True, use -dy/dt instead of de/dt. Default: False.
        N_filter: Derivative filter coefficient. 0 means no filter. Default: 0.
    """

    def __init__(self, Kp, Ki, Kd, dt, u_min=-np.inf, u_max=np.inf,
                 derivative_on_measurement=False, N_filter=0):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.dt = dt
        self.u_min = u_min
        self.u_max = u_max
        self.derivative_on_measurement = derivative_on_measurement
        self.N_filter = N_filter

        self._integral = 0.0
        self._prev_error = 0.0
        self._prev_measurement = 0.0
        self._prev_derivative = 0.0
        self._first_call = True

    def compute(self, error, measurement=0.0):
        """
        Compute PID control output for the current error.

        Args:
            error: Current tracking error e = r - y. Scalar.
            measurement: Current plant output y (used if derivative_on_measurement=True). Scalar.

        Returns:
            u: Control output. Scalar.
        """
        # Proportional term
        P = self.Kp * error

        # Integral term (accumulated before clamping check)
        self._integral += error * self.dt
        I = self.Ki * self._integral

        # Derivative term
        if self._first_call:
            raw_derivative = 0.0
            self._first_call = False
        else:
            if self.derivative_on_measurement:
                raw_derivative = -(measurement - self._prev_measurement) / self.dt
            else:
                raw_derivative = (error - self._prev_error) / self.dt

        # Optional low-pass filter on derivative
        if self.N_filter > 0:
            alpha = self.dt * self.N_filter / (1.0 + self.dt * self.N_filter)
            filtered_derivative = alpha * raw_derivative + (1.0 - alpha) * self._prev_derivative
        else:
            filtered_derivative = raw_derivative

        D = self.Kd * filtered_derivative
        self._prev_derivative = filtered_derivative

        # Compute total output
        u = P + I + D

        # Anti-windup: clamp output and back-calculate integral
        u_clamped = np.clip(u, self.u_min, self.u_max)
        if u != u_clamped:
            # Back-calculate: remove the excess integral
            self._integral -= error * self.dt

        self._prev_error = error
        self._prev_measurement = measurement

        return u_clamped

    def reset(self):
        """
        Reset the controller's internal state (integral, derivative memory).
        """
        self._integral = 0.0
        self._prev_error = 0.0
        self._prev_measurement = 0.0
        self._prev_derivative = 0.0
        self._first_call = True

In [ ]:
# --- Show integral action eliminates steady-state error ---
# --- Show derivative action reduces overshoot ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) P-only vs PI: integral eliminates steady-state error
Kp_test = 10.0

# P-only
def p_only(error, state, t):
    return Kp_test * error

t_p, x_p, _ = simulate_plant(mass_spring_damper, p_only, R_STEP, T=T_SIM)

# PI
pi_ctrl = PIDController(Kp=Kp_test, Ki=5.0, Kd=0.0, dt=DT)
def pi_controller(error, state, t):
    return pi_ctrl.compute(error)

t_pi, x_pi, _ = simulate_plant(mass_spring_damper, pi_controller, R_STEP, T=T_SIM)

axes[0].plot(t_p, x_p[:, 0], color=COLORS['blue'], label='P only')
axes[0].plot(t_pi, x_pi[:, 0], color=COLORS['red'], label='PI')
axes[0].axhline(y=R_STEP, color='black', linestyle='--', alpha=0.5, label='Reference')
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Position $x(t)$')
axes[0].set_title('Integral Action Eliminates Steady-State Error')
axes[0].legend()

# (b) PI vs PID: derivative reduces overshoot
pi_ctrl2 = PIDController(Kp=20.0, Ki=10.0, Kd=0.0, dt=DT)
def pi_ctrl2_fn(error, state, t):
    return pi_ctrl2.compute(error)

pid_ctrl = PIDController(Kp=20.0, Ki=10.0, Kd=5.0, dt=DT)
def pid_ctrl_fn(error, state, t):
    return pid_ctrl.compute(error)

t_pi2, x_pi2, _ = simulate_plant(mass_spring_damper, pi_ctrl2_fn, R_STEP, T=T_SIM)
t_pid, x_pid, _ = simulate_plant(mass_spring_damper, pid_ctrl_fn, R_STEP, T=T_SIM)

axes[1].plot(t_pi2, x_pi2[:, 0], color=COLORS['red'], label='PI (Kp=20, Ki=10)')
axes[1].plot(t_pid, x_pid[:, 0], color=COLORS['green'], label='PID (Kp=20, Ki=10, Kd=5)')
axes[1].axhline(y=R_STEP, color='black', linestyle='--', alpha=0.5, label='Reference')
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Position $x(t)$')
axes[1].set_title('Derivative Action Reduces Overshoot')
axes[1].legend(fontsize=10)

# (c) Control effort comparison
pid_ctrl3 = PIDController(Kp=20.0, Ki=10.0, Kd=5.0, dt=DT)
def pid_ctrl3_fn(error, state, t):
    return pid_ctrl3.compute(error)

t_pid3, x_pid3, u_pid3 = simulate_plant(mass_spring_damper, pid_ctrl3_fn, R_STEP, T=T_SIM)

axes[2].plot(t_pid3, u_pid3, color=COLORS['green'], label='PID control effort')
axes[2].set_xlabel('Time [s]')
axes[2].set_ylabel('Control $u(t)$')
axes[2].set_title('PID Control Effort')
axes[2].legend()

plt.tight_layout()
plt.show()

# Verification
ess_p = R_STEP - x_p[-1, 0]
ess_pi = R_STEP - x_pi[-1, 0]
status_pi = "PASS" if abs(ess_pi) < 0.01 else "FAIL"
print(f"P-only steady-state error: {ess_p:.4f}")
print(f"PI steady-state error: {ess_pi:.6f} [{status_pi}] (should be ~0)")

overshoot_pi2 = (np.max(x_pi2[:, 0]) - R_STEP) / R_STEP * 100
overshoot_pid = (np.max(x_pid[:, 0]) - R_STEP) / R_STEP * 100
status_d = "PASS" if overshoot_pid < overshoot_pi2 else "FAIL"
print(f"PI overshoot: {overshoot_pi2:.2f}%")
print(f"PID overshoot: {overshoot_pid:.2f}% [{status_d}] (should be less than PI)")

---
## 4. Stability Analysis

### Closed-Loop Transfer Function

For the mass-spring-damper plant $G(s) = \frac{1}{ms^2 + bs + k}$ with PID controller $C(s) = K_p + \frac{K_i}{s} + K_d s$:

$$T(s) = \frac{C(s)G(s)}{1 + C(s)G(s)} = \frac{K_d s^2 + K_p s + K_i}{ms^3 + (b + K_d)s^2 + (k + K_p)s + K_i}$$

The characteristic equation is:

$$\boxed{ms^3 + (b + K_d)s^2 + (k + K_p)s + K_i = 0}$$

### Routh-Hurwitz Stability Criterion

For the polynomial $a_3 s^3 + a_2 s^2 + a_1 s + a_0 = 0$ with $a_3 = m$, $a_2 = b + K_d$, $a_1 = k + K_p$, $a_0 = K_i$, the Routh array is:

| | $s^3$ | $s^1$ |
|---|---|---|
| Row 1 | $a_3 = m$ | $a_1 = k + K_p$ |
| Row 2 | $a_2 = b + K_d$ | $a_0 = K_i$ |
| Row 3 | $\frac{a_2 a_1 - a_3 a_0}{a_2}$ | 0 |
| Row 4 | $K_i$ | |

For stability, all elements in the first column must be positive:

1. $m > 0$ (always true)
2. $b + K_d > 0$ (true for $K_d > -b$)
3. $\frac{(b + K_d)(k + K_p) - m K_i}{b + K_d} > 0$
4. $K_i > 0$

Condition 3 simplifies to:

$$\boxed{K_i < \frac{(b + K_d)(k + K_p)}{m}}$$

This defines a **stability boundary** in the $(K_p, K_i)$ plane for fixed $K_d$.

In [ ]:
def routh_hurwitz_stable(Kp, Ki, Kd, m=MSD_MASS, b=MSD_DAMPING, k=MSD_SPRING):
    """
    Check Routh-Hurwitz stability for PID + 2nd-order plant.

    The characteristic equation is:
        m*s^3 + (b+Kd)*s^2 + (k+Kp)*s + Ki = 0

    Args:
        Kp: Proportional gain.
        Ki: Integral gain.
        Kd: Derivative gain.
        m, b, k: Plant parameters.

    Returns:
        stable: Boolean, True if all Routh conditions satisfied.
    """
    a3 = m
    a2 = b + Kd
    a1 = k + Kp
    a0 = Ki

    if a2 <= 0:
        return False
    if a0 <= 0:
        return False
    if a1 <= 0:
        return False
    # Row 3 element: (a2*a1 - a3*a0) / a2 > 0
    if (a2 * a1 - a3 * a0) <= 0:
        return False
    return True


def check_simulation_stability(plant_func, Kp, Ki, Kd, dt=DT, T=15.0):
    """
    Check stability empirically by simulating and seeing if output diverges.

    Args:
        plant_func: Plant dynamics function.
        Kp, Ki, Kd: PID gains.
        dt: Time step [s].
        T: Simulation time [s].

    Returns:
        stable: Boolean, True if output remains bounded.
    """
    pid = PIDController(Kp=Kp, Ki=Ki, Kd=Kd, dt=dt)

    def ctrl(error, state, t):
        return pid.compute(error)

    _, x, _ = simulate_plant(plant_func, ctrl, R_STEP, dt=dt, T=T)
    # Check if output stays bounded
    return np.all(np.abs(x[:, 0]) < 1e6)


# --- Plot stability regions in (Kp, Ki) plane ---
KD_FIXED = 2.0
Kp_range = np.linspace(0.1, 50.0, 200)
Ki_range = np.linspace(0.1, 150.0, 200)
Kp_grid, Ki_grid = np.meshgrid(Kp_range, Ki_range)

# Routh-Hurwitz boundary: Ki = (b + Kd)(k + Kp) / m
Ki_boundary = (MSD_DAMPING + KD_FIXED) * (MSD_SPRING + Kp_range) / MSD_MASS

# Stability map from Routh-Hurwitz
stability_map = np.zeros_like(Kp_grid)
for i in range(len(Ki_range)):
    for j in range(len(Kp_range)):
        stability_map[i, j] = routh_hurwitz_stable(Kp_grid[i, j], Ki_grid[i, j], KD_FIXED)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: stability region
axes[0].contourf(Kp_grid, Ki_grid, stability_map, levels=[-0.5, 0.5, 1.5],
                 colors=[COLORS['red'], COLORS['green']], alpha=0.3)
axes[0].plot(Kp_range, Ki_boundary, 'k-', linewidth=2, label='Routh-Hurwitz boundary')
axes[0].set_xlabel('$K_p$')
axes[0].set_ylabel('$K_i$')
axes[0].set_title(f'Stability Region ($K_d = {KD_FIXED}$)')
axes[0].legend()

# Mark test points
test_points = [
    (10.0, 10.0, 'A'),   # Stable
    (5.0, 50.0, 'B'),    # Unstable
    (30.0, 60.0, 'C'),   # Stable
    (5.0, 20.0, 'D'),    # Near boundary
]

for kp, ki, label in test_points:
    rh_stable = routh_hurwitz_stable(kp, ki, KD_FIXED)
    marker = 'o' if rh_stable else 'x'
    color = COLORS['green'] if rh_stable else COLORS['red']
    axes[0].plot(kp, ki, marker, color=color, markersize=12, markeredgewidth=3)
    axes[0].annotate(label, (kp, ki), textcoords="offset points",
                     xytext=(8, 8), fontsize=14, fontweight='bold')

# Right: simulate at test points
for kp, ki, label in test_points:
    pid = PIDController(Kp=kp, Ki=ki, Kd=KD_FIXED, dt=DT)
    def ctrl(error, state, t, _pid=pid):
        return _pid.compute(error)
    t_sim, x_sim, _ = simulate_plant(mass_spring_damper, ctrl, R_STEP, T=8.0)
    # Clip for plotting
    x_plot = np.clip(x_sim[:, 0], -5, 5)
    axes[1].plot(t_sim, x_plot, linewidth=1.5,
                 label=f'{label}: $K_p$={kp}, $K_i$={ki}')

axes[1].axhline(y=R_STEP, color='black', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Position $x(t)$')
axes[1].set_title('Simulation Verification')
axes[1].legend(fontsize=9)
axes[1].set_ylim(-5, 5)

plt.tight_layout()
plt.show()

# Verification: compare Routh-Hurwitz prediction with simulation
print(f"Stability verification (Kd = {KD_FIXED}):")
for kp, ki, label in test_points:
    rh_stable = routh_hurwitz_stable(kp, ki, KD_FIXED)
    sim_stable = check_simulation_stability(mass_spring_damper, kp, ki, KD_FIXED)
    match = rh_stable == sim_stable
    status = "PASS" if match else "FAIL"
    print(f"  Point {label} (Kp={kp:5.1f}, Ki={ki:5.1f}): "
          f"RH={'stable' if rh_stable else 'unstable':>8s}, "
          f"Sim={'stable' if sim_stable else 'unstable':>8s} [{status}]")

---
## 5. Tuning Methods: Ziegler-Nichols

### The Ultimate Gain Method

The Ziegler-Nichols (ZN) method is a classical heuristic for tuning PID controllers **without a plant model**:

**Procedure:**
1. Set $K_i = 0$, $K_d = 0$ (P-only control)
2. Increase $K_p$ from zero until the system exhibits **sustained oscillation** (marginal stability)
3. Record the **ultimate gain** $K_u$ and the **ultimate period** $T_u$
4. Compute PID gains from the table:

| Controller | $K_p$ | $K_i$ | $K_d$ |
|-----------|-------|-------|-------|
| P | $0.5 K_u$ | 0 | 0 |
| PI | $0.45 K_u$ | $\frac{2 K_p}{T_u}$ | 0 |
| PID | $0.6 K_u$ | $\frac{2 K_p}{T_u}$ | $\frac{K_p T_u}{8}$ |

### Theoretical Ultimate Gain

For the mass-spring-damper with P-only control, the characteristic equation is:

$$ms^2 + bs + (k + K_p) = 0$$

Substituting $s = j\omega$ for sustained oscillation:

$$-m\omega^2 + jb\omega + (k + K_u) = 0$$

Real part: $k + K_u - m\omega^2 = 0$ and imaginary part: $b\omega = 0$.

Since $b \neq 0$, a second-order system with damping never reaches pure sustained oscillation under P-only control. In practice, we look for the gain that produces the least-damped response and use a modified approach. For our demonstration, we use a **PID + plant** characteristic equation $ms^3 + (b+K_d)s^2 + (k+K_p)s + K_i = 0$ with $K_i = 0, K_d = 0$, and treat the marginal stability condition from the Routh criterion.

In [ ]:
def find_ultimate_gain(plant_func, dt=DT, T=20.0, kp_range=None):
    """
    Find the ultimate gain Ku and period Tu using binary search.

    Strategy: For a 2nd-order plant, we add a small integral gain to create
    a 3rd-order characteristic equation, then find Kp that makes the system
    marginally stable.

    For m*s^3 + b*s^2 + (k+Kp)*s + Ki = 0, with small Ki, the Routh boundary is:
        Ki_crit = b*(k+Kp)/m
    We fix Ki and sweep Kp to find oscillation.

    Args:
        plant_func: Plant dynamics function.
        dt: Time step [s].
        T: Simulation time for each trial [s].
        kp_range: Tuple (kp_low, kp_high) for search. Default: (0.1, 200).

    Returns:
        Ku: Ultimate gain.
        Tu: Ultimate period [s].
    """
    if kp_range is None:
        kp_range = (0.1, 200.0)

    # Use a small integral gain to create 3rd order system
    Ki_probe = 1.0

    def is_unstable(kp):
        pid = PIDController(Kp=kp, Ki=Ki_probe, Kd=0.0, dt=dt)
        def ctrl(error, state, t):
            return pid.compute(error)
        _, x, _ = simulate_plant(plant_func, ctrl, R_STEP, dt=dt, T=T)
        # Check if oscillations grow in the last portion
        last_quarter = x[3 * len(x) // 4:, 0]
        first_quarter = x[len(x) // 4:len(x) // 2, 0]
        amp_late = np.max(np.abs(last_quarter - R_STEP))
        return amp_late > 1e3 or np.any(np.isnan(x))

    # Binary search for ultimate gain
    kp_low, kp_high = kp_range
    for _ in range(50):
        kp_mid = (kp_low + kp_high) / 2
        if is_unstable(kp_mid):
            kp_high = kp_mid
        else:
            kp_low = kp_mid

    Ku = (kp_low + kp_high) / 2

    # Find Tu: simulate at Ku and measure oscillation period
    pid = PIDController(Kp=Ku, Ki=Ki_probe, Kd=0.0, dt=dt)
    def ctrl(error, state, t):
        return pid.compute(error)
    t_sim, x_sim, _ = simulate_plant(plant_func, ctrl, R_STEP, dt=dt, T=T)

    # Find period from zero crossings of (y - reference) in the last half
    y = x_sim[:, 0]
    half = len(y) // 2
    y_centered = y[half:] - R_STEP
    zero_crossings = []
    for i in range(len(y_centered) - 1):
        if y_centered[i] * y_centered[i + 1] < 0:
            zero_crossings.append(half + i)

    if len(zero_crossings) >= 3:
        # Period = 2 * (average half-period)
        periods = []
        for i in range(len(zero_crossings) - 2):
            periods.append((zero_crossings[i + 2] - zero_crossings[i]) * dt)
        Tu = np.mean(periods)
    else:
        # Fallback: use natural frequency estimate
        Tu = 2 * np.pi / np.sqrt(MSD_SPRING / MSD_MASS)

    return Ku, Tu


def ziegler_nichols_tune(plant_func, dt=DT, T=20.0):
    """
    Auto-tune PID gains using Ziegler-Nichols ultimate gain method.

    Args:
        plant_func: Plant dynamics function.
        dt: Time step [s].
        T: Simulation time for each trial [s].

    Returns:
        Kp, Ki, Kd: Tuned PID gains.
        Ku: Ultimate gain.
        Tu: Ultimate period [s].
    """
    Ku, Tu = find_ultimate_gain(plant_func, dt=dt, T=T)

    # Ziegler-Nichols PID tuning rules
    Kp = 0.6 * Ku
    Ki = 2.0 * Kp / Tu
    Kd = Kp * Tu / 8.0

    return Kp, Ki, Kd, Ku, Tu


# --- Run ZN tuning ---
Kp_zn, Ki_zn, Kd_zn, Ku, Tu = ziegler_nichols_tune(mass_spring_damper)
print(f"Ziegler-Nichols Auto-Tuning Results:")
print(f"  Ultimate gain Ku = {Ku:.4f}")
print(f"  Ultimate period Tu = {Tu:.4f} s")
print(f"  PID gains: Kp = {Kp_zn:.4f}, Ki = {Ki_zn:.4f}, Kd = {Kd_zn:.4f}")

In [ ]:
# --- Compare ZN-tuned vs hand-tuned PID ---

# Hand-tuned gains (chosen for good response)
KP_HAND = 20.0
KI_HAND = 10.0
KD_HAND = 5.0

# ZN-tuned controller
pid_zn = PIDController(Kp=Kp_zn, Ki=Ki_zn, Kd=Kd_zn, dt=DT)
def zn_ctrl(error, state, t):
    return pid_zn.compute(error)

# Hand-tuned controller
pid_hand = PIDController(Kp=KP_HAND, Ki=KI_HAND, Kd=KD_HAND, dt=DT)
def hand_ctrl(error, state, t):
    return pid_hand.compute(error)

t_zn, x_zn, u_zn = simulate_plant(mass_spring_damper, zn_ctrl, R_STEP, T=T_SIM)
t_hand, x_hand, u_hand = simulate_plant(mass_spring_damper, hand_ctrl, R_STEP, T=T_SIM)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t_zn, x_zn[:, 0], color=COLORS['blue'],
             label=f'ZN (Kp={Kp_zn:.1f}, Ki={Ki_zn:.1f}, Kd={Kd_zn:.1f})')
axes[0].plot(t_hand, x_hand[:, 0], color=COLORS['red'],
             label=f'Hand (Kp={KP_HAND}, Ki={KI_HAND}, Kd={KD_HAND})')
axes[0].axhline(y=R_STEP, color='black', linestyle='--', alpha=0.5, label='Reference')
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Position $x(t)$')
axes[0].set_title('Step Response: ZN-Tuned vs Hand-Tuned')
axes[0].legend(fontsize=10)

axes[1].plot(t_zn, u_zn, color=COLORS['blue'], label='ZN control')
axes[1].plot(t_hand, u_hand, color=COLORS['red'], label='Hand-tuned control')
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Control $u(t)$')
axes[1].set_title('Control Effort Comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

# Performance metrics
def compute_metrics(t, x, r):
    """Compute step response metrics: overshoot, settling time, steady-state error."""
    y = x[:, 0]
    overshoot = (np.max(y) - r) / r * 100 if np.max(y) > r else 0.0
    # Settling time: last time |y - r| > 2% of r
    settled = np.abs(y - r) < 0.02 * abs(r)
    if np.any(settled):
        # Find last time it was NOT settled
        not_settled = np.where(~settled)[0]
        if len(not_settled) > 0:
            ts = t[not_settled[-1]]
        else:
            ts = 0.0
    else:
        ts = t[-1]
    ess = abs(r - y[-1])
    return overshoot, ts, ess

os_zn, ts_zn, ess_zn = compute_metrics(t_zn, x_zn, R_STEP)
os_hand, ts_hand, ess_hand = compute_metrics(t_hand, x_hand, R_STEP)

print(f"\nPerformance Comparison:")
print(f"{'Metric':<25s} {'ZN-Tuned':>12s} {'Hand-Tuned':>12s}")
print(f"{'-'*50}")
print(f"{'Overshoot [%]':<25s} {os_zn:>12.2f} {os_hand:>12.2f}")
print(f"{'Settling time [s]':<25s} {ts_zn:>12.3f} {ts_hand:>12.3f}")
print(f"{'Steady-state error':<25s} {ess_zn:>12.6f} {ess_hand:>12.6f}")

---
## 6. Application 1: Cart-Pole Balancing

### Nonlinear Dynamics

The cart-pole (inverted pendulum on a cart) has state $\mathbf{x} = [x, \dot{x}, \theta, \dot{\theta}]^T$ where $x$ is cart position and $\theta$ is pole angle from vertical. The control input is horizontal force $F$ on the cart.

The equations of motion (derived via Lagrangian mechanics):

$$\ddot{x} = \frac{F + ml\dot{\theta}^2\sin\theta - mg\sin\theta\cos\theta}{M + m\sin^2\theta}$$

$$\ddot{\theta} = \frac{(M+m)g\sin\theta - (F + ml\dot{\theta}^2\sin\theta)\cos\theta}{l(M + m\sin^2\theta)}$$

### Linearized Dynamics (about $\theta = 0$)

For small angles ($\sin\theta \approx \theta$, $\cos\theta \approx 1$, $\dot{\theta}^2 \approx 0$):

$$\ddot{x} \approx \frac{F - mg\theta}{M}$$

$$\ddot{\theta} \approx \frac{(M+m)g\theta - F}{Ml}$$

### Control Strategy

We use a PID controller on the pole angle $\theta$ to stabilize the pendulum upright. The controller outputs force $F$:

$$F = \text{PID}(\theta_{\text{ref}} - \theta)$$

with $\theta_{\text{ref}} = 0$ (upright).

In [ ]:
def cart_pole_dynamics(state, u):
    """
    Nonlinear cart-pole dynamics.

    State: [x, x_dot, theta, theta_dot]
    Control: u = horizontal force F on cart.

    Args:
        state: [x, x_dot, theta, theta_dot]. Shape: (4,)
        u: Horizontal force [N]. Scalar.

    Returns:
        state_dot: [x_dot, x_ddot, theta_dot, theta_ddot]. Shape: (4,)
    """
    x, x_dot, theta, theta_dot = state
    M = CP_CART_MASS
    m = CP_POLE_MASS
    l = CP_POLE_LENGTH
    g = CP_GRAVITY
    F = u

    sin_th = np.sin(theta)
    cos_th = np.cos(theta)
    denom = M + m * sin_th**2

    x_ddot = (F + m * l * theta_dot**2 * sin_th - m * g * sin_th * cos_th) / denom
    theta_ddot = ((M + m) * g * sin_th - (F + m * l * theta_dot**2 * sin_th) * cos_th) / (l * denom)

    return np.array([x_dot, x_ddot, theta_dot, theta_ddot])


def simulate_cart_pole(pid_theta, theta0=0.1, dt=DT, T=5.0):
    """
    Simulate cart-pole with PID angle control.

    Args:
        pid_theta: PIDController instance for angle control.
        theta0: Initial pole angle [rad]. Default: 0.1 (about 5.7 degrees).
        dt: Time step [s].
        T: Simulation time [s].

    Returns:
        t_hist: Time array. Shape: (M,)
        x_hist: State history [x, x_dot, theta, theta_dot]. Shape: (M, 4)
        u_hist: Control force history. Shape: (M,)
    """
    n_steps = int(T / dt)
    state = np.array([0.0, 0.0, theta0, 0.0])

    t_hist = np.zeros(n_steps + 1)
    x_hist = np.zeros((n_steps + 1, 4))
    u_hist = np.zeros(n_steps + 1)

    x_hist[0] = state
    pid_theta.reset()

    for i in range(n_steps):
        theta = state[2]
        error_theta = 0.0 - theta  # Reference angle is 0 (upright)
        F = pid_theta.compute(error_theta, measurement=theta)
        u_hist[i] = F

        state = rk4_step(cart_pole_dynamics, state, F, dt)
        t_hist[i + 1] = (i + 1) * dt
        x_hist[i + 1] = state

    u_hist[-1] = u_hist[-2]
    return t_hist, x_hist, u_hist


# --- Simulate cart-pole balancing ---
# PID tuned for cart-pole angle stabilization
CP_KP = 50.0
CP_KI = 10.0
CP_KD = 15.0
THETA0 = 0.2  # Initial angle ~11.5 degrees

pid_cp = PIDController(Kp=CP_KP, Ki=CP_KI, Kd=CP_KD, dt=DT)
t_cp, x_cp, u_cp = simulate_cart_pole(pid_cp, theta0=THETA0, T=5.0)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Angle
axes[0, 0].plot(t_cp, np.degrees(x_cp[:, 2]), color=COLORS['blue'])
axes[0, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0, 0].set_xlabel('Time [s]')
axes[0, 0].set_ylabel('Angle [deg]')
axes[0, 0].set_title('Pole Angle $\\theta(t)$')

# Angular velocity
axes[0, 1].plot(t_cp, np.degrees(x_cp[:, 3]), color=COLORS['red'])
axes[0, 1].set_xlabel('Time [s]')
axes[0, 1].set_ylabel('Angular velocity [deg/s]')
axes[0, 1].set_title('Angular Velocity $\\dot{\\theta}(t)$')

# Cart position
axes[1, 0].plot(t_cp, x_cp[:, 0], color=COLORS['green'])
axes[1, 0].set_xlabel('Time [s]')
axes[1, 0].set_ylabel('Cart position [m]')
axes[1, 0].set_title('Cart Position $x(t)$')

# Control force
axes[1, 1].plot(t_cp, u_cp, color=COLORS['yellow'])
axes[1, 1].set_xlabel('Time [s]')
axes[1, 1].set_ylabel('Force [N]')
axes[1, 1].set_title('Control Force $F(t)$')

plt.suptitle(f'Cart-Pole Balancing (Kp={CP_KP}, Ki={CP_KI}, Kd={CP_KD}, '
             f'$\\theta_0$={np.degrees(THETA0):.1f} deg)', fontsize=14)
plt.tight_layout()
plt.show()

# Stability verification
final_angle = np.degrees(abs(x_cp[-1, 2]))
max_angle = np.degrees(np.max(np.abs(x_cp[:, 2])))
stable = final_angle < 1.0  # Within 1 degree of upright
status = "PASS" if stable else "FAIL"
print(f"Cart-pole stabilization: final angle = {final_angle:.4f} deg [{status}]")
print(f"  Max angle excursion: {max_angle:.2f} deg")
print(f"  Final cart position: {x_cp[-1, 0]:.4f} m")

---
## 7. Application 2: Drone Altitude Hold

### 1D Vertical Dynamics

A quadrotor in the vertical direction obeys:

$$m\ddot{z} = T - mg$$

where $z$ is altitude, $T$ is total thrust, and $g$ is gravitational acceleration.

### PID Altitude Controller

The thrust command is:

$$T = mg + u_{\text{PID}}(z_{\text{ref}} - z)$$

The $mg$ term is a **feedforward** component that compensates for gravity, and the PID term handles tracking and disturbance rejection.

### Disturbance: Wind Gust

We model a wind gust as a step disturbance force $d(t)$ applied at $t = 3$ s:

$$m\ddot{z} = T - mg + d(t)$$

In [ ]:
def drone_altitude_dynamics(state, u, m=DRONE_MASS, g=DRONE_GRAVITY, disturbance=0.0):
    """
    1D vertical drone dynamics: m*z'' = T - m*g + d.

    Args:
        state: [z, z_dot]. Shape: (2,)
        u: Thrust T [N]. Scalar.
        m: Drone mass [kg].
        g: Gravitational acceleration [m/s^2].
        disturbance: External disturbance force [N]. Scalar.

    Returns:
        state_dot: [z_dot, z_ddot]. Shape: (2,)
    """
    z, z_dot = state
    z_ddot = (u - m * g + disturbance) / m
    return np.array([z_dot, z_ddot])


def simulate_drone(pid_alt, z_ref, z0=0.0, dt=DT, T=T_SIM,
                   gust_time=3.0, gust_force=-2.0):
    """
    Simulate drone altitude hold with PID and wind gust disturbance.

    Args:
        pid_alt: PIDController instance for altitude.
        z_ref: Reference altitude [m].
        z0: Initial altitude [m].
        dt: Time step [s].
        T: Simulation time [s].
        gust_time: Time at which gust starts [s].
        gust_force: Gust force magnitude [N] (negative = downward).

    Returns:
        t_hist: Time array. Shape: (M,)
        z_hist: State history [z, z_dot]. Shape: (M, 2)
        T_hist: Thrust command history. Shape: (M,)
    """
    n_steps = int(T / dt)
    state = np.array([z0, 0.0])
    m = DRONE_MASS
    g = DRONE_GRAVITY

    t_hist = np.zeros(n_steps + 1)
    z_hist = np.zeros((n_steps + 1, 2))
    T_hist = np.zeros(n_steps + 1)

    z_hist[0] = state
    pid_alt.reset()

    for i in range(n_steps):
        t = i * dt
        z = state[0]
        error = z_ref - z

        # PID output + gravity feedforward
        u_pid = pid_alt.compute(error, measurement=z)
        thrust = m * g + u_pid  # Feedforward + feedback
        thrust = max(thrust, 0.0)  # Thrust cannot be negative
        T_hist[i] = thrust

        # Wind gust disturbance
        d = gust_force if t >= gust_time else 0.0

        # Dynamics with disturbance
        def drone_dyn(s, u_in):
            return drone_altitude_dynamics(s, u_in, disturbance=d)

        state = rk4_step(drone_dyn, state, thrust, dt)
        t_hist[i + 1] = (i + 1) * dt
        z_hist[i + 1] = state

    T_hist[-1] = T_hist[-2]
    return t_hist, z_hist, T_hist


# --- Simulate drone altitude hold ---
Z_REF = 5.0     # Target altitude [m]
GUST_TIME = 3.0  # Wind gust start time [s]
GUST_FORCE = -3.0  # Downward gust [N]

# PID for altitude
pid_drone = PIDController(Kp=15.0, Ki=5.0, Kd=8.0, dt=DT)
t_dr, z_dr, T_dr = simulate_drone(pid_drone, Z_REF, z0=0.0,
                                    gust_time=GUST_TIME, gust_force=GUST_FORCE)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Altitude tracking
axes[0].plot(t_dr, z_dr[:, 0], color=COLORS['blue'], label='Altitude $z(t)$')
axes[0].axhline(y=Z_REF, color='black', linestyle='--', alpha=0.5, label=f'Reference ({Z_REF} m)')
axes[0].axvline(x=GUST_TIME, color=COLORS['red'], linestyle=':', alpha=0.7, label='Gust onset')
axes[0].fill_between([GUST_TIME, t_dr[-1]], -1, Z_REF + 2,
                     color=COLORS['red'], alpha=0.05)
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Altitude [m]')
axes[0].set_title('Drone Altitude Hold with Wind Gust')
axes[0].legend()

# Thrust command
axes[1].plot(t_dr, T_dr, color=COLORS['green'], label='Thrust $T(t)$')
axes[1].axhline(y=DRONE_MASS * DRONE_GRAVITY, color='black', linestyle='--',
               alpha=0.5, label=f'Hover thrust ({DRONE_MASS * DRONE_GRAVITY:.2f} N)')
axes[1].axvline(x=GUST_TIME, color=COLORS['red'], linestyle=':', alpha=0.7, label='Gust onset')
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Thrust [N]')
axes[1].set_title('Thrust Command')
axes[1].legend()

plt.tight_layout()
plt.show()

# Verification
final_alt = z_dr[-1, 0]
alt_error = abs(final_alt - Z_REF)
recovered = alt_error < 0.05
status = "PASS" if recovered else "FAIL"
print(f"Drone altitude hold: final altitude = {final_alt:.4f} m, error = {alt_error:.4f} m [{status}]")

# Max altitude drop during gust
gust_start_idx = int(GUST_TIME / DT)
min_alt_after_gust = np.min(z_dr[gust_start_idx:, 0])
max_drop = Z_REF - min_alt_after_gust
print(f"Max altitude drop during gust: {max_drop:.4f} m")

---
## 8. Anti-Windup and Derivative Filtering

### The Integral Windup Problem

When the actuator **saturates** (e.g., a motor has maximum torque), the integral term continues to accumulate error even though the control output is clamped. When the error changes sign, the integral must "unwind" before the controller responds — causing large overshoot and slow recovery.

### Anti-Windup: Clamping

The simplest anti-windup strategy: **stop integrating** when the output is saturated. Our `PIDController` class implements this by reverting the integral update when the output is clamped.

### Derivative Kick

When the reference $r(t)$ has a step change, $\dot{e} = \dot{r} - \dot{y}$ produces a large spike ("derivative kick"). Solution: compute the derivative on the **measurement** instead:

$$D = -K_d \frac{dy}{dt}$$

Since the reference step does not appear in $y$ instantaneously, this eliminates the kick.

### Derivative Low-Pass Filtering

High-frequency noise in the measurement is amplified by the derivative. A first-order filter:

$$D_{\text{filtered}}(s) = \frac{K_d s}{1 + \frac{s}{N}}$$

where $N$ controls the filter cutoff (typical: $N = 10$ to $20$). In discrete time with time constant $\tau = 1/N$:

$$D[k] = \alpha \cdot D_{\text{raw}}[k] + (1 - \alpha) \cdot D[k-1], \quad \alpha = \frac{\Delta t \cdot N}{1 + \Delta t \cdot N}$$

In [ ]:
# --- Demonstrate integral windup and anti-windup ---

# Saturated actuator: force limited to [-5, 5]
U_MAX = 5.0

# PID without anti-windup (manually disable by using large limits internally,
# then clamp output externally)
pid_no_aw = PIDController(Kp=10.0, Ki=8.0, Kd=2.0, dt=DT,
                           u_min=-np.inf, u_max=np.inf)  # No internal clamping

def ctrl_no_aw(error, state, t):
    u_raw = pid_no_aw.compute(error)
    return np.clip(u_raw, -U_MAX, U_MAX)  # External saturation only

# PID with anti-windup
pid_aw = PIDController(Kp=10.0, Ki=8.0, Kd=2.0, dt=DT,
                        u_min=-U_MAX, u_max=U_MAX)  # Internal clamping

def ctrl_aw(error, state, t):
    return pid_aw.compute(error)

t_no_aw, x_no_aw, u_no_aw = simulate_plant(mass_spring_damper, ctrl_no_aw, R_STEP, T=15.0)
t_aw, x_aw, u_aw = simulate_plant(mass_spring_damper, ctrl_aw, R_STEP, T=15.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t_no_aw, x_no_aw[:, 0], color=COLORS['red'],
             label='Without anti-windup')
axes[0].plot(t_aw, x_aw[:, 0], color=COLORS['green'],
             label='With anti-windup')
axes[0].axhline(y=R_STEP, color='black', linestyle='--', alpha=0.5, label='Reference')
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Position $x(t)$')
axes[0].set_title(f'Integral Windup Effect (|u| $\\leq$ {U_MAX})')
axes[0].legend()

axes[1].plot(t_no_aw, u_no_aw, color=COLORS['red'],
             label='Without anti-windup', alpha=0.7)
axes[1].plot(t_aw, u_aw, color=COLORS['green'],
             label='With anti-windup', alpha=0.7)
axes[1].axhline(y=U_MAX, color='black', linestyle=':', alpha=0.5)
axes[1].axhline(y=-U_MAX, color='black', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Control $u(t)$')
axes[1].set_title('Control Effort')
axes[1].legend()

plt.tight_layout()
plt.show()

os_no_aw = (np.max(x_no_aw[:, 0]) - R_STEP) / R_STEP * 100
os_aw = (np.max(x_aw[:, 0]) - R_STEP) / R_STEP * 100
status_aw = "PASS" if os_aw < os_no_aw else "FAIL"
print(f"Overshoot without anti-windup: {os_no_aw:.2f}%")
print(f"Overshoot with anti-windup: {os_aw:.2f}% [{status_aw}]")

In [ ]:
# --- Derivative kick and derivative filtering ---

# Standard derivative (on error)
pid_standard = PIDController(Kp=15.0, Ki=5.0, Kd=8.0, dt=DT,
                              derivative_on_measurement=False, N_filter=0)

# Derivative on measurement
pid_dom = PIDController(Kp=15.0, Ki=5.0, Kd=8.0, dt=DT,
                         derivative_on_measurement=True, N_filter=0)

# Derivative on measurement + filter
pid_filtered = PIDController(Kp=15.0, Ki=5.0, Kd=8.0, dt=DT,
                              derivative_on_measurement=True, N_filter=20)

# Simulate all three
def make_ctrl(pid):
    def ctrl(error, state, t, _pid=pid):
        return _pid.compute(error, measurement=state[0])
    return ctrl

t_std, x_std, u_std = simulate_plant(mass_spring_damper, make_ctrl(pid_standard), R_STEP, T=T_SIM)
t_dom, x_dom, u_dom = simulate_plant(mass_spring_damper, make_ctrl(pid_dom), R_STEP, T=T_SIM)
t_flt, x_flt, u_flt = simulate_plant(mass_spring_damper, make_ctrl(pid_filtered), R_STEP, T=T_SIM)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Step response
axes[0].plot(t_std, x_std[:, 0], color=COLORS['blue'], label='Standard PID')
axes[0].plot(t_dom, x_dom[:, 0], color=COLORS['red'], label='D on measurement')
axes[0].plot(t_flt, x_flt[:, 0], color=COLORS['green'], label='D on meas. + filter (N=20)')
axes[0].axhline(y=R_STEP, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Position $x(t)$')
axes[0].set_title('Step Response: Derivative Improvements')
axes[0].legend(fontsize=10)

# Control effort (zoom into first 0.1s to see derivative kick)
t_zoom = 0.05  # seconds
n_zoom = int(t_zoom / DT)
axes[1].plot(t_std[:n_zoom], u_std[:n_zoom], color=COLORS['blue'], label='Standard PID')
axes[1].plot(t_dom[:n_zoom], u_dom[:n_zoom], color=COLORS['red'], label='D on measurement')
axes[1].plot(t_flt[:n_zoom], u_flt[:n_zoom], color=COLORS['green'], label='D on meas. + filter')
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Control $u(t)$')
axes[1].set_title('Control Effort at Step (Derivative Kick)')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

# Verify: derivative on measurement should have smaller initial control spike
max_u_std = np.max(np.abs(u_std[:n_zoom]))
max_u_dom = np.max(np.abs(u_dom[:n_zoom]))
status_dk = "PASS" if max_u_dom < max_u_std else "FAIL"
print(f"Max initial control (standard): {max_u_std:.2f}")
print(f"Max initial control (D on measurement): {max_u_dom:.2f} [{status_dk}] (should be smaller)")

---
## 9. Comprehensive Visualizations

In [ ]:
# === PANEL 1: P vs PI vs PID Step Response on Mass-Spring-Damper ===

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- (a) Step Response Comparison ---
KP_COMP = 15.0
KI_COMP = 8.0
KD_COMP = 4.0

# P-only
def p_ctrl_comp(error, state, t):
    return KP_COMP * error

# PI
pid_pi_comp = PIDController(Kp=KP_COMP, Ki=KI_COMP, Kd=0.0, dt=DT)
def pi_ctrl_comp(error, state, t):
    return pid_pi_comp.compute(error)

# PID
pid_pid_comp = PIDController(Kp=KP_COMP, Ki=KI_COMP, Kd=KD_COMP, dt=DT)
def pid_ctrl_comp(error, state, t):
    return pid_pid_comp.compute(error)

t_p_c, x_p_c, _ = simulate_plant(mass_spring_damper, p_ctrl_comp, R_STEP, T=T_SIM)
t_pi_c, x_pi_c, _ = simulate_plant(mass_spring_damper, pi_ctrl_comp, R_STEP, T=T_SIM)
t_pid_c, x_pid_c, _ = simulate_plant(mass_spring_damper, pid_ctrl_comp, R_STEP, T=T_SIM)

axes[0, 0].plot(t_p_c, x_p_c[:, 0], color=COLORS['blue'], label='P only')
axes[0, 0].plot(t_pi_c, x_pi_c[:, 0], color=COLORS['red'], label='PI')
axes[0, 0].plot(t_pid_c, x_pid_c[:, 0], color=COLORS['green'], label='PID')
axes[0, 0].axhline(y=R_STEP, color='black', linestyle='--', alpha=0.5, label='Reference')
axes[0, 0].set_xlabel('Time [s]')
axes[0, 0].set_ylabel('Position $x(t)$')
axes[0, 0].set_title('(a) Step Response: P vs PI vs PID')
axes[0, 0].legend(fontsize=10)

# --- (b) Root Locus-Style: Closed-Loop Poles as Kp Varies ---
# For PID+plant with fixed Ki, Kd, characteristic equation:
#   m*s^3 + (b+Kd)*s^2 + (k+Kp)*s + Ki = 0
KI_RL = 5.0
KD_RL = 2.0
kp_sweep = np.linspace(0.1, 100.0, 200)

all_poles = []
for kp in kp_sweep:
    coeffs = [MSD_MASS, MSD_DAMPING + KD_RL, MSD_SPRING + kp, KI_RL]
    poles = np.roots(coeffs)
    all_poles.append(poles)
all_poles = np.array(all_poles)

for j in range(all_poles.shape[1]):
    scatter = axes[0, 1].scatter(all_poles[:, j].real, all_poles[:, j].imag,
                                  c=kp_sweep, cmap='viridis', s=3, alpha=0.6)

axes[0, 1].axvline(x=0, color='black', linestyle='-', alpha=0.3)
axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[0, 1].set_xlabel('Real')
axes[0, 1].set_ylabel('Imaginary')
axes[0, 1].set_title(f'(b) Closed-Loop Poles vs $K_p$ ($K_i$={KI_RL}, $K_d$={KD_RL})')
cbar = plt.colorbar(scatter, ax=axes[0, 1])
cbar.set_label('$K_p$')

# --- (c) Gain Sensitivity: Sweep Kp, Plot Overshoot & Settling Time ---
kp_sens = np.linspace(1.0, 80.0, 40)
overshoots = []
settling_times = []

for kp in kp_sens:
    pid_test = PIDController(Kp=kp, Ki=5.0, Kd=3.0, dt=DT)
    def ctrl_test(error, state, t, _pid=pid_test):
        return _pid.compute(error)
    t_test, x_test, _ = simulate_plant(mass_spring_damper, ctrl_test, R_STEP, T=T_SIM)
    os_val, ts_val, _ = compute_metrics(t_test, x_test, R_STEP)
    overshoots.append(os_val)
    settling_times.append(ts_val)

ax_os = axes[1, 0]
ax_ts = ax_os.twinx()
line1, = ax_os.plot(kp_sens, overshoots, color=COLORS['blue'], label='Overshoot [%]')
line2, = ax_ts.plot(kp_sens, settling_times, color=COLORS['red'], label='Settling time [s]')
ax_os.set_xlabel('$K_p$')
ax_os.set_ylabel('Overshoot [%]', color=COLORS['blue'])
ax_ts.set_ylabel('Settling time [s]', color=COLORS['red'])
axes[1, 0].set_title('(c) Gain Sensitivity: $K_p$ Sweep')
lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax_os.legend(lines, labels, loc='upper right', fontsize=10)

# --- (d) Cart-Pole Animation Frames ---
# Draw stick figure at several time steps
n_frames = 8
frame_indices = np.linspace(0, len(t_cp) - 1, n_frames, dtype=int)
cart_w, cart_h = 0.3, 0.15
pole_length = 2 * CP_POLE_LENGTH  # Full pole length for visualization

ax_cp = axes[1, 1]
ax_cp.set_aspect('equal')

alphas = np.linspace(0.15, 1.0, n_frames)
for idx, fi in enumerate(frame_indices):
    x_cart = x_cp[fi, 0]
    theta = x_cp[fi, 2]
    alpha = alphas[idx]

    # Cart (rectangle)
    cart_rect = plt.Rectangle((x_cart - cart_w / 2, -cart_h / 2), cart_w, cart_h,
                              facecolor=COLORS['blue'], alpha=alpha, edgecolor='black',
                              linewidth=1)
    ax_cp.add_patch(cart_rect)

    # Pole
    pole_x = x_cart + pole_length * np.sin(theta)
    pole_y = pole_length * np.cos(theta)
    ax_cp.plot([x_cart, pole_x], [0, pole_y], 'o-',
              color=COLORS['red'], alpha=alpha, linewidth=2.5, markersize=5)

    # Time label
    if idx % 2 == 0:
        ax_cp.text(x_cart, -0.25, f't={t_cp[fi]:.2f}s',
                  ha='center', fontsize=7, alpha=alpha)

# Ground line
x_min = np.min(x_cp[:, 0]) - 1
x_max = np.max(x_cp[:, 0]) + 1
ax_cp.plot([x_min, x_max], [-cart_h / 2, -cart_h / 2], 'k-', linewidth=1)
ax_cp.set_xlim(x_min, x_max)
ax_cp.set_ylim(-0.5, 1.5)
ax_cp.set_xlabel('Cart position [m]')
ax_cp.set_ylabel('Height [m]')
ax_cp.set_title('(d) Cart-Pole Stabilization Frames')

plt.tight_layout()
plt.show()

In [ ]:
# === PANEL 2: ZN vs Hand-Tuned Convergence ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Track error convergence over time
# ZN
pid_zn2 = PIDController(Kp=Kp_zn, Ki=Ki_zn, Kd=Kd_zn, dt=DT)
def zn_ctrl2(error, state, t):
    return pid_zn2.compute(error)
t_zn2, x_zn2, _ = simulate_plant(mass_spring_damper, zn_ctrl2, R_STEP, T=T_SIM)

# Hand-tuned
pid_hand2 = PIDController(Kp=KP_HAND, Ki=KI_HAND, Kd=KD_HAND, dt=DT)
def hand_ctrl2(error, state, t):
    return pid_hand2.compute(error)
t_hand2, x_hand2, _ = simulate_plant(mass_spring_damper, hand_ctrl2, R_STEP, T=T_SIM)

# Error magnitude (log scale)
err_zn = np.abs(R_STEP - x_zn2[:, 0])
err_hand = np.abs(R_STEP - x_hand2[:, 0])

# Smooth for plotting (moving average)
window = 500
err_zn_smooth = np.convolve(err_zn, np.ones(window) / window, mode='valid')
err_hand_smooth = np.convolve(err_hand, np.ones(window) / window, mode='valid')
t_smooth = t_zn2[:len(err_zn_smooth)]

axes[0].semilogy(t_smooth, err_zn_smooth, color=COLORS['blue'], label='ZN-tuned')
axes[0].semilogy(t_smooth, err_hand_smooth, color=COLORS['red'], label='Hand-tuned')
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('$|e(t)|$ (log scale)')
axes[0].set_title('Error Convergence: ZN vs Hand-Tuned')
axes[0].legend()

# Cumulative absolute error (IAE)
iae_zn = np.cumsum(np.abs(R_STEP - x_zn2[:, 0])) * DT
iae_hand = np.cumsum(np.abs(R_STEP - x_hand2[:, 0])) * DT

axes[1].plot(t_zn2, iae_zn, color=COLORS['blue'], label=f'ZN (IAE={iae_zn[-1]:.3f})')
axes[1].plot(t_hand2, iae_hand, color=COLORS['red'], label=f'Hand (IAE={iae_hand[-1]:.3f})')
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Integrated Absolute Error')
axes[1].set_title('Cumulative IAE Comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Final IAE - ZN: {iae_zn[-1]:.4f}, Hand: {iae_hand[-1]:.4f}")

---
## 10. Extensions and Further Reading

### Cascade Control

In cascade control, an **outer loop** PID generates a reference for an **inner loop** PID. For example, in the cart-pole:
- Outer loop: PID on cart position $x$ produces reference angle $\theta_{\text{ref}}$
- Inner loop: PID on angle $\theta$ produces force $F$

The inner loop rejects disturbances faster and improves overall performance. This is standard in drone control (altitude $\to$ velocity $\to$ thrust).

### Feedforward Control

PID is purely reactive. **Feedforward** adds a model-based term:

$$u = u_{\text{ff}}(r) + u_{\text{PID}}(e)$$

For example, the gravity compensation $mg$ in the drone controller is a feedforward term. More generally, if we know the plant model $G(s)$, the ideal feedforward is $G^{-1}(s) \cdot r(s)$.

### Limitations of PID

| Limitation | Description |
|-----------|------------|
| Nonlinear systems | PID assumes linearity; performance degrades for large operating ranges |
| MIMO systems | PID is SISO; multi-input multi-output systems need decoupling or MIMO controllers |
| Delay-dominant plants | Large dead time requires Smith predictor or model predictive control |
| Optimality | PID does not optimize any cost function; LQR provides optimal gains |
| Constraint handling | PID cannot systematically handle state/input constraints (use MPC) |

### Connection to Optimal Control (LQR)

For linear systems, the **Linear Quadratic Regulator** (LQR) computes the optimal state-feedback gain $K$ that minimizes:

$$J = \int_0^\infty \left(\mathbf{x}^T Q \mathbf{x} + u^T R \, u\right) dt$$

For a second-order system, LQR produces gains equivalent to a PD controller. Adding an integral state to the system and applying LQR gives optimal PID-like gains. See the companion notebook on **LQR control** for a detailed treatment.

### Recommended Next Steps

1. **LQR Control** — optimal state-feedback for linear systems
2. **Model Predictive Control (MPC)** — optimization-based control with constraints
3. **Nonlinear Control** — sliding mode, feedback linearization
4. **System Identification** — learning plant models from data for model-based tuning